Cell 1 - Cài đặt dbt và xác thực Google Cloud

In [17]:
!pip install dbt-core dbt-bigquery -q

from google.colab import auth
auth.authenticate_user()

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/.config/application_default_credentials.json"
print("Xác thực thành công!")

Xác thực thành công!


Cell 2 - Tạo project dbt_telco

In [18]:
import os

# Tạo thư mục
os.makedirs("/content/dbt_telco/models/staging", exist_ok=True)

# ----- profiles.yml -----
profiles = """
dbt_telco:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: btqt2-telco-dbt-bigquery
      dataset: silver_telco
      location: asia-southeast1
      threads: 4
      timeout_seconds: 300
      priority: interactive
"""
with open("/content/dbt_telco/profiles.yml", "w") as f:
    f.write(profiles)

# ----- dbt_project.yml -----
dbt_project = """
name: 'dbt_telco'
version: '1.0.0'
config-version: 2

profile: 'dbt_telco'

model-paths: ["models"]
clean-targets:
  - "target"
  - "dbt_packages"

models:
  dbt_telco:
    +materialized: table
    staging:
      +materialized: view
"""
with open("/content/dbt_telco/dbt_project.yml", "w") as f:
    f.write(dbt_project)

print("Đã tạo cấu trúc project!")

Đã tạo cấu trúc project!


Cell 3 - Khai báo nguồn Bronze và tạo model stg_telco_customers

In [19]:
# ----- _sources.yml -----
sources = """
version: 2

sources:
  - name: bronze_telco
    database: btqt2-telco-dbt-bigquery
    schema: bronze_telco
    tables:
      - name: customers_raw
        description: "Raw customer churn data loaded from CSV into Bronze layer"
"""
with open("/content/dbt_telco/models/staging/_sources.yml", "w") as f:
    f.write(sources)

# ----- stg_telco_customers.sql -----
sql = """
with source as (
    select *
    from {{ source('bronze_telco', 'customers_raw') }}
),

renamed_and_casted as (
    select
        trim(customerID)                                    as customer_id,
        trim(gender)                                        as gender,
        cast(SeniorCitizen as int64)                        as senior_citizen,
        trim(Partner)                                       as partner,
        trim(Dependents)                                    as dependents,
        cast(tenure as int64)                               as tenure,
        trim(PhoneService)                                  as phone_service,
        trim(MultipleLines)                                 as multiple_lines,
        trim(InternetService)                               as internet_service,
        trim(OnlineSecurity)                                as online_security,
        trim(OnlineBackup)                                  as online_backup,
        trim(DeviceProtection)                              as device_protection,
        trim(TechSupport)                                   as tech_support,
        trim(StreamingTV)                                   as streaming_tv,
        trim(StreamingMovies)                               as streaming_movies,
        trim(Contract)                                      as contract,
        trim(PaperlessBilling)                              as paperless_billing,
        trim(PaymentMethod)                                 as payment_method,
        cast(MonthlyCharges as numeric)                     as monthly_charges,
        safe_cast(
            nullif(trim(TotalCharges), '') as numeric
        )                                                   as total_charges,
        trim(Churn)                                         as churn,
        _ingested_at,
        _source_file
    from source
)

select *
from renamed_and_casted
"""
with open("/content/dbt_telco/models/staging/stg_telco_customers.sql", "w") as f:
    f.write(sql)

print("Đã tạo model staging!")

Đã tạo model staging!


Cell 4 - Kiểm tra cấu trúc thư mục models

In [20]:
!rm -f /content/dbt_telco/models/sources.yml
!ls -R /content/dbt_telco/models

/content/dbt_telco/models:
staging

/content/dbt_telco/models/staging:
_sources.yml  stg_telco_customers.sql


Cell 5 - Kiểm tra kết nối và chạy model

In [21]:
!dbt debug --project-dir /content/dbt_telco --profiles-dir /content/dbt_telco
!dbt run --select stg_telco_customers --project-dir /content/dbt_telco --profiles-dir /content/dbt_telco

03:40:44  Running with dbt=1.12.6
03:40:44  dbt version: 1.12.6
03:40:44  python version: 3.13.16
03:40:44  python path: /usr/bin/python3
03:40:44  os info: Linux-6.6.122+-x86_64-with-glibc2.39
03:40:47  Using profiles dir at /content/dbt_telco
03:40:47  Using profiles.yml file at /content/dbt_telco/profiles.yml
03:40:47  Using dbt_project.yml file at /content/dbt_telco/dbt_project.yml
03:40:47  adapter type: bigquery
03:40:47  adapter version: 1.12.1
03:40:47  Configuration:
03:40:47    profiles.yml file [OK found and valid]
03:40:47    dbt_project.yml file [OK found and valid]
03:40:47  Required dependencies:
03:40:47   - git [OK found]

03:40:47  Connection:
03:40:47    method: oauth
03:40:47    database: btqt2-telco-dbt-bigquery
03:40:47    execution_project: btqt2-telco-dbt-bigquery
03:40:47    schema: silver_telco
03:40:47    location: asia-southeast1
03:40:47    priority: interactive
03:40:47    maximum_bytes_billed: None
03:40:47    reservation: None
03:40:47    impersonate_ser

Cell 6 - Kiểm tra số dòng và 11 dòng tenure = 0

In [22]:
from google.cloud import bigquery
bq = bigquery.Client(project="btqt2-telco-dbt-bigquery", location="asia-southeast1")

# Số dòng
print(bq.query("""
SELECT COUNT(*) AS so_dong
FROM `btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers`
""").to_dataframe())

# 11 dòng tenure = 0
print(bq.query("""
SELECT customer_id, tenure, monthly_charges, total_charges
FROM `btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers`
WHERE tenure = 0
""").to_dataframe())

   so_dong
0     7043
   customer_id  tenure monthly_charges total_charges
0   2923-ARZLG       0    19.700000000          None
1   7644-OMVMY       0    19.850000000          None
2   2520-SGTTA       0    20.000000000          None
3   3115-CZMZD       0    20.250000000          None
4   3213-VVOLG       0    25.350000000          None
5   4367-NUYAO       0    25.750000000          None
6   4472-LVYGI       0    52.550000000          None
7   1371-DWPAZ       0    56.050000000          None
8   2775-SEFEE       0    61.900000000          None
9   4075-WKNIU       0    73.350000000          None
10  5709-LVOEQ       0    80.850000000          None


Cell 7 - Đối chiếu chỉ số tổng hợp và kiểu dữ liệu của view

In [23]:
from google.cloud import bigquery

bq = bigquery.Client(project="btqt2-telco-dbt-bigquery", location="asia-southeast1")

# 5.1 Chỉ số tổng hợp (so với Bảng 3.2 của báo cáo)
print(bq.query("""
SELECT
  COUNT(*)                          AS so_dong,
  COUNT(DISTINCT customer_id)       AS so_customer_id,
  COUNTIF(total_charges IS NULL)    AS total_charges_null,
  ROUND(SUM(monthly_charges), 2)    AS tong_monthly_charges,
  COUNTIF(churn = 'Yes')            AS churn_yes,
  COUNTIF(churn = 'No')             AS churn_no
FROM `btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers`
""").to_dataframe())

# 5.2 Kiểu dữ liệu của view (dùng làm minh chứng cho mục 4.2)
print(bq.query("""
SELECT column_name, data_type
FROM `btqt2-telco-dbt-bigquery.silver_telco.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'stg_telco_customers'
ORDER BY ordinal_position
""").to_dataframe())

   so_dong  so_customer_id  total_charges_null tong_monthly_charges  \
0     7043            7043                  11     456116.600000000   

   churn_yes  churn_no  
0       1869      5174  
          column_name  data_type
0         customer_id     STRING
1              gender     STRING
2      senior_citizen      INT64
3             partner     STRING
4          dependents     STRING
5              tenure      INT64
6       phone_service     STRING
7      multiple_lines     STRING
8    internet_service     STRING
9     online_security     STRING
10      online_backup     STRING
11  device_protection     STRING
12       tech_support     STRING
13       streaming_tv     STRING
14   streaming_movies     STRING
15           contract     STRING
16  paperless_billing     STRING
17     payment_method     STRING
18    monthly_charges    NUMERIC
19      total_charges    NUMERIC
20              churn     STRING
21       _ingested_at  TIMESTAMP
22       _source_file     STRING


Cell 8 - Đóng gói project dbt_telco.zip

In [24]:
!zip -r dbt_telco.zip dbt_telco
from google.colab import files
files.download("dbt_telco.zip")

updating: dbt_telco/ (stored 0%)
updating: dbt_telco/profiles.yml (deflated 39%)
updating: dbt_telco/models/ (stored 0%)
updating: dbt_telco/models/staging/ (stored 0%)
updating: dbt_telco/models/staging/_sources.yml (deflated 35%)
updating: dbt_telco/models/staging/stg_telco_customers.sql (deflated 73%)
updating: dbt_telco/logs/ (stored 0%)
updating: dbt_telco/logs/dbt.log (deflated 85%)
updating: dbt_telco/.user.yml (stored 0%)
updating: dbt_telco/target/ (stored 0%)
updating: dbt_telco/target/run_results.json (deflated 63%)
updating: dbt_telco/target/compiled/ (stored 0%)
updating: dbt_telco/target/compiled/dbt_telco/ (stored 0%)
updating: dbt_telco/target/compiled/dbt_telco/models/ (stored 0%)
updating: dbt_telco/target/compiled/dbt_telco/models/staging/ (stored 0%)
updating: dbt_telco/target/compiled/dbt_telco/models/staging/stg_telco_customers.sql (deflated 73%)
updating: dbt_telco/target/osi_document.json (deflated 21%)
updating: dbt_telco/target/partial_parse.msgpack (deflated 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>